# 04 - Eleição do vencedor e promoção controlada

**Etapa 5** | Fluxo: passos 7, 8 e 9

1. **Vencedor do ciclo:** maior média de F1 macro na validação cruzada (empate técnico < 0,002 favorece o modelo A).
2. **Confronto com o vigente:** o vencedor e o vigente são avaliados **uma vez** no **mesmo conjunto de teste** do ciclo
   (o teste nunca entrou em treino, votação ou escolha do vencedor).
3. **Regra de promoção (`config.yaml`):** promove se `CV(vencedor) >= 0,60` **e** `F1_teste(vencedor) - F1_teste(vigente) >= 0,005`.
   Sem vigente, promove o primeiro modelo. Caso contrário **mantém o vigente**.
4. Tudo é registrado em `models/modelo_vigente.json`, no MLflow e em `evidencias/<cycle_id>/decisao.json`.
   Modelos anteriores permanecem em `models/historico/`.

A decisão usa apenas o manifesto e o modelo versionados no repositório, não depende de arquivos do runner anterior.

In [1]:
# Preparação: localiza a raiz do projeto (o nbconvert executa com a pasta notebooks/ como diretório atual)
import sys, json, warnings
from pathlib import Path
RAIZ = Path.cwd().resolve()
if RAIZ.name == "notebooks":
    RAIZ = RAIZ.parent
sys.path.insert(0, str(RAIZ))
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mlflow

from src.config import carregar_config, p, dir_, ler_json, gravar_json
from src import ciclo, rastreio

cfg = carregar_config()
pd.set_option("display.width", 160, "display.max_columns", 30)

In [2]:
from datetime import datetime, timezone
from src import features as ft, modelagem as mo, promocao as pr
ctx = ciclo.carregar_contexto(cfg)
alvo = cfg["dados"]["alvo"]
proc = dir_(cfg, "dir_processed")
teste = pd.read_csv(proc / "teste.csv")
X_cols = ft.COLS_NUM + ft.COLS_CAT

res = ler_json(proc / "resultados_cv.json")
assert res["cycle_id"] == ctx["cycle_id"] and res["data_version"] == ctx["data_version"], "resultados de outro ciclo!"
assert set(res["modelos"]) == {"modelo_A", "modelo_B"}, "os dois algoritmos precisam ter sido treinados"

eleicao = pr.eleger_vencedor(cfg, res["cv"])
print("Vencedor do ciclo:", eleicao)

Vencedor do ciclo: {'vencedor': 'modelo_B', 'motivo': 'maior média de CV', 'cv_A': 0.6717159011501028, 'cv_B': 0.679246774604465, 'metrica': 'f1_macro'}


In [3]:
chave = eleicao["vencedor"]
bundle = mo.carregar_modelo(p(res["modelos"][chave]["caminho"]))
info_vencedor = {"vencedor": chave, "algoritmo": res["modelos"][chave]["algoritmo"],
                 "features": bundle["features"], "run_id": res["modelos"][chave]["run_id"]}
m_teste_venc = mo.avaliar_pipeline(bundle["pipeline"], teste[X_cols], teste[alvo])

manifesto_anterior = pr.ler_manifesto()
m_teste_vig = None
if manifesto_anterior:
    vig = mo.carregar_modelo(p(manifesto_anterior["caminho_modelo"]))
    m_teste_vig = mo.avaliar_pipeline(vig["pipeline"], teste[X_cols], teste[alvo])   # mesmo teste, mesmo protocolo

metrica = cfg["avaliacao"]["metrica"]
decisao = pr.decidir_promocao(cfg, res["cv"][chave]["media"][metrica], m_teste_venc[metrica],
                              m_teste_vig[metrica] if m_teste_vig else None)
print(json.dumps(decisao, indent=2, ensure_ascii=False))
print("vigente anterior:", manifesto_anterior["modelo_id"] if manifesto_anterior else "(nenhum)")

{
  "piso_metrica_cv": 0.6,
  "limiar_melhoria": 0.005,
  "cv_vencedor": 0.679246774604465,
  "teste_vencedor": 0.6804356321253207,
  "teste_vigente": null,
  "delta_teste": null,
  "decisao": "promovido",
  "motivo": "não havia modelo vigente (primeiro ciclo)"
}
vigente anterior: (nenhum)


In [4]:
ctx["fim_utc"] = datetime.now(timezone.utc).isoformat(timespec="seconds")
if decisao["decisao"] == "promovido":
    manifesto = pr.promover(cfg, ctx, p(res["modelos"][chave]["caminho"]), info_vencedor, decisao)
else:
    manifesto = pr.manter(ctx, decisao)
print(f"DECISÃO: {decisao['decisao'].upper()} | vigente agora: {manifesto['modelo_id']}")
print(json.dumps(manifesto, indent=2, ensure_ascii=False))

DECISÃO: PROMOVIDO | vigente agora: C001_20261124T234632Z_random_forest
{
  "modelo_id": "C001_20261124T234632Z_random_forest",
  "algoritmo": "random_forest",
  "chave": "modelo_B",
  "cycle_id": "C001_20261124T234632Z",
  "data_version": "v001_8c382cd5af",
  "features": [
    "num__tempo_contrato_meses",
    "cat__tipo_contrato_mensal",
    "num__mensalidade",
    "num__chamados_suporte_3m",
    "num__chamados_por_ano",
    "cat__tipo_contrato_bianual",
    "num__atraso_pagamentos_6m",
    "num__gasto_por_servico",
    "num__tem_atraso",
    "num__satisfacao"
  ],
  "metrica": "f1_macro",
  "valor_cv": 0.679246774604465,
  "valor_teste": 0.6804356321253207,
  "run_id": "1d53a57a31684d42abff378a9456ab5e",
  "caminho_modelo": "models/historico/C001_20261124T234632Z_random_forest.joblib",
  "promovido_em_utc": "2026-11-24T23:46:49+00:00",
  "decisao": "promovido",
  "motivo": "não havia modelo vigente (primeiro ciclo)",
  "modelo_anterior": null,
  "ultima_avaliacao": {
    "cycle_id": 

## Registro no MLflow, evidência auditável e histórico dos ciclos

In [5]:
ev = ciclo.dir_evidencias(ctx); ev.mkdir(parents=True, exist_ok=True)
auditoria = {
    "cycle_id": ctx["cycle_id"], "data_version": ctx["data_version"], "gatilho": ctx["gatilho"],
    "github_run_id": ctx["github_run_id"], "github_run_url": ctx["github_run_url"],
    "features_eleitas": res["features"], "eleicao_vencedor": eleicao,
    "cv": {k: res["cv"][k]["media"] for k in res["cv"]},
    "metricas_teste_vencedor": m_teste_venc, "metricas_teste_vigente_anterior": m_teste_vig,
    "decisao": decisao, "modelo_vigente_apos_ciclo": manifesto["modelo_id"],
    "run_ids": ctx["run_ids"], "drift": ctx.get("drift"),
}
gravar_json(ev / "decisao.json", auditoria)

rastreio.configurar(cfg)
with rastreio.run("selecao_promocao", ctx, etapa="decisao",
                  tags_extra={"decisao": decisao["decisao"], "vencedor": chave, "run_id_vencedor": info_vencedor["run_id"],
                              "modelo_vigente_id": manifesto["modelo_id"]}) as r:
    mlflow.log_params({"limiar_melhoria": cfg["promocao"]["limiar_melhoria"], "piso_metrica_cv": cfg["promocao"]["piso_metrica_cv"],
                       "metrica_principal": metrica, "direcao": cfg["avaliacao"]["direcao"]})
    mets = {"cv_vencedor": decisao["cv_vencedor"], "teste_vencedor": decisao["teste_vencedor"],
            "cv_A": eleicao["cv_A"], "cv_B": eleicao["cv_B"], "promovido": 1.0 if decisao["decisao"] == "promovido" else 0.0}
    if m_teste_vig:
        mets.update(teste_vigente_anterior=decisao["teste_vigente"], delta_teste=decisao["delta_teste"])
    mlflow.log_metrics(mets)
    mlflow.log_artifact(str(ev / "decisao.json"))
    mlflow.log_artifact(str(p("models", "modelo_vigente.json")))
    ctx["run_ids"]["decisao"] = r.info.run_id

# Histórico entre ciclos (versionado no Git): é a última escrita, só ocorre se tudo acima deu certo
linha = {
    "cycle_id": ctx["cycle_id"], "timestamp_utc": ctx["fim_utc"], "gatilho": ctx["gatilho"], "github_run_id": ctx["github_run_id"],
    "data_version": ctx["data_version"], "n_lotes": ctx["n_lotes"], "n_linhas": ctx["n_linhas"],
    "n_features_eleitas": len(res["features"]), "features_eleitas": "|".join(res["features"]),
    "vencedor": chave, "algoritmo_vencedor": info_vencedor["algoritmo"],
    "f1_cv_A": round(eleicao["cv_A"], 4), "f1_cv_B": round(eleicao["cv_B"], 4),
    "f1_teste_vencedor": round(decisao["teste_vencedor"], 4),
    "f1_teste_vigente_anterior": None if decisao["teste_vigente"] is None else round(decisao["teste_vigente"], 4),
    "delta_teste": None if decisao["delta_teste"] is None else round(decisao["delta_teste"], 4),
    "decisao": decisao["decisao"], "motivo": decisao["motivo"], "modelo_vigente_id": manifesto["modelo_id"],
    "drift_alerta": bool(ctx.get("drift", {}).get("alerta", False)), "max_psi": round(ctx.get("drift", {}).get("max_psi", 0.0), 4),
}
hist = pd.concat([ciclo.ler_historico(), pd.DataFrame([linha])], ignore_index=True)
hist[ciclo.COLUNAS_HISTORICO].to_csv(ciclo.caminho_historico(), index=False)
ciclo.salvar_contexto(cfg, ctx)
hist[["cycle_id", "data_version", "vencedor", "f1_cv_A", "f1_cv_B", "f1_teste_vencedor", "f1_teste_vigente_anterior", "decisao", "drift_alerta"]]

,cycle_id,data_version,vencedor,f1_cv_A,f1_cv_B,f1_teste_vencedor,f1_teste_vigente_anterior,decisao,drift_alerta
0,C001_20261124T234632Z,v001_8c382cd5af,modelo_B,0.6717,0.6792,0.6804,None,promovido,False
